In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Users/anuragsaha1712@gmail.com/consolidated_pipeline/FCMG_DB/1_setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
# Define schema names (normally loaded from utilities notebook)
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "gross_price", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportsbar-anu/{data_source}/*.csv'
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"

bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

In [0]:
print(base_path)
print(landing_path)
print(processed_path)

In [0]:
# Read from processed directory since landing is empty - files have already been processed
correct_processed_path = processed_path.replace("/*.csv", "")
df = spark.read.options(header = True , inferSchema = True).csv(f"{correct_processed_path}*.csv").withColumn("read_timestamp", F.current_timestamp()).select("*","_metadata.file_name","_metadata.file_size")

print("Total Rows:" , df.count())
df.show(5)


In [0]:
display(df.limit(20))

In [0]:
df.write\
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("append") \
    .saveAsTable(bronze_table)

In [0]:
# Remove /*.csv from paths before using with dbutils.fs operations
correct_landing_path = landing_path.replace("/*.csv", "")
correct_processed_path = processed_path.replace("/*.csv", "")

files = dbutils.fs.ls(correct_landing_path)

for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{correct_processed_path}{file_info.name}",
        True
    )

In [0]:
df_orders = spark.sql(f"SELECT * FROM {bronze_table}")

In [0]:
# Standardize order_placement_date to YYYY-MM-DD format
# Handle multiple input formats: YYYY/MM/DD, DD/MM/YYYY, DD-MM-YYYY, and full text dates

from pyspark.sql.functions import col, coalesce, when, regexp_replace, try_to_date

# First, create a temporary column with the original values for comparison
df_temp = df_orders.withColumn("original_date", col("order_placement_date"))

# Use try_to_date to handle invalid formats gracefully (returns NULL instead of throwing error)
# Replace the order_placement_date column with the standardized version
df_cleaned = df_temp.withColumn(
    "order_placement_date",
    coalesce(
        # Format 1: YYYY/MM/DD
        try_to_date(col("order_placement_date"), "yyyy/MM/dd"),
        
        # Format 2: YYYY-MM-DD
        try_to_date(col("order_placement_date"), "yyyy-MM-dd"),
        
        # Format 3: DD/MM/YYYY
        try_to_date(col("order_placement_date"), "dd/MM/yyyy"),
        
        # Format 4: DD-MM-YYYY
        try_to_date(col("order_placement_date"), "dd-MM-yyyy"),
        
        # Format 5: Full text format like "Monday, July 07, 2025"
        # Extract everything after the first comma and space, then parse
        try_to_date(regexp_replace(col("order_placement_date"), "^[A-Za-z]+,\\s*", ""), "MMMM dd, yyyy")
    )
).withColumn(
    # Add a flag for records that couldn't be parsed
    "date_parse_failed",
    when(col("order_placement_date").isNull(), True).otherwise(False)
)

# Show before and after comparison
print("Before and After date standardization:")
df_cleaned.select(
    "original_date", 
    "order_placement_date", 
    "date_parse_failed"
).show(20, truncate=False)

# Check if any dates failed to parse
failed_count = df_cleaned.filter(col("date_parse_failed") == True).count()
print(f"\nTotal rows: {df_cleaned.count()}")
print(f"Failed to parse: {failed_count}")

if failed_count > 0:
    print("\nSample of failed dates:")
    df_cleaned.filter(col("date_parse_failed") == True).select("original_date").distinct().show(10, truncate=False)
else:
    print("\nAll dates successfully parsed and standardized to YYYY-MM-DD format!")

# Drop the temporary original_date column
df_cleaned = df_cleaned.drop("original_date")

In [0]:
display(df_cleaned.limit(20))


In [0]:
df_orders = df_cleaned.filter(F.col("order_qty").isNotNull())

In [0]:
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"),F.col("customer_id"))
    .otherwise("999999")
    .cast("string")
)

In [0]:
display(df_orders.limit(20))

In [0]:
df_products = spark.table("fmcg.silver.products")

display(df_products.limit(20))

In [0]:
df_joined = df_orders.join(df_products,on = "product_id" , how = "inner").select(df_orders["*"],df_products["*"])
display(df_joined.limit(10))

In [0]:
# Resolve duplicate columns from join (both DataFrames share product_id, read_timestamp, file_name, file_size)
df_joined = df_orders.join(df_products, on="product_id", how="inner").select(
    df_orders["*"],
    "product_code", "division", "category", "product", "variant"
)

if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed" , "true"
    ).option("mergeSchema","true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark,silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"),"silver.order_placement = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

### Gold Layer

In [0]:
df_gold = spark.sql(f"SELECT order_id , order_placement_date as date , customer_id as customer_code , product_id , order_qty as sold_quantity FROM {silver_table};")

df_gold.show(5)

In [0]:
gold_table

In [0]:
if not (spark.catalog.tableExists(gold_table)):
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed" , "true"
    ).option("mergeSchema","true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark,gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"),"source.date = gold.date AND order_id = AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

### Merge with parent company